# LeetCode 375 — Guess Number Higher or Lower II bằng Python

Link đề: https://leetcode.com/problems/guess-number-higher-or-lower-ii/ · Độ khó: **Medium** · Chủ đề: Minimax, Math, Dynamic Programming, Game Theory

Skeleton LeetCode:

```python
class Solution:
    def getMoneyAmount(self, n: int) -> int:
        ...
```

Ghi chú: đề được viết lại bằng tiếng Việt theo đề gốc trên LeetCode; mọi đáp án trong notebook đều đã được kiểm chứng bằng code (kể cả bằng một "oracle" vét cạn độc lập).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đây **không** phải trò đoán số quen thuộc (chia đôi, mỗi lần đoán tốn 1 lượt) mà là trò đoán số có **giá tiền**: đoán sai số `x` thì **mất `x` đô**; ta cần số tiền **tối thiểu** để **chắc chắn thắng trong mọi trường hợp xấu nhất**.
2. Nắm khái niệm **minimax** (cực tiểu hóa cái xấu nhất) và cách biến nó thành công thức đệ quy `f(lo, hi) = min over k of ( k + max(f(lo, k-1), f(k+1, hi)) )`.
3. Dùng các công cụ Python: danh sách 2 chiều, `min`/`max` với generator, `functools.lru_cache`, vòng lặp theo **độ dài đoạn**, đệ quy.
4. Đi từ **đệ quy vét cạn (hàm mũ)** → **đệ quy có nhớ (memo)** → **quy hoạch động bottom-up trên đoạn `[i, j]`** → **cắt nhánh (branch and bound)**, và biết tính độ phức tạp `O(n³)`.
5. Tránh các lỗi kinh điển: nhầm minimax với "chia đôi", nhầm `max` thành `min`/`+`, tràn chỉ số `dp[k+1][j]`, tạo bảng 2D sai bằng `[[0]*m]*n`, đặt sai cơ sở `dp[i][i]`.

## 2. Đề bài & ví dụ

Ta chơi một trò đoán số như sau:

1. Đối thủ chọn một số nguyên trong khoảng `1..n` (ta không biết).
2. Ta đoán một số.
3. Nếu đoán đúng, **ta thắng** và trò chơi kết thúc.
4. Nếu đoán sai, đối thủ cho biết số thật **lớn hơn** hay **nhỏ hơn** số ta đoán, và ta tiếp tục đoán.
5. **Mỗi lần đoán sai số `x`, ta phải trả `x` đô.** Hết tiền thì thua.

Cho `n`, trả về **số tiền tối thiểu** cần có để **đảm bảo thắng bất kể đối thủ chọn số nào**.

**Ví dụ 1:** `n = 10` → `16`. (Chiến lược: đoán 7; nếu số lớn hơn → xét `[8,10]` đoán 9; nếu số nhỏ hơn → xét `[1,6]` đoán 3, ... Trường hợp tệ nhất trả `16`.)

**Ví dụ 2:** `n = 1` → `0` (chỉ có một số, đoán là trúng, không mất gì).

**Ví dụ 3:** `n = 2` → `1` (đoán 1: nếu sai mất 1 đô, rồi biết chắc là 2).

**Ràng buộc:** `1 <= n <= 200`.

In [1]:
import functools
import itertools
import random
import time
from typing import List

examples = [(10, 16), (1, 0), (2, 1)]
for n, expected in examples:
    print(f"n = {n:>2} -> đáp án mong đợi {expected}")

n = 10 -> đáp án mong đợi 16
n =  1 -> đáp án mong đợi 0
n =  2 -> đáp án mong đợi 1


## 3. Kiến thức nền

### 3.1. "Trường hợp xấu nhất" và `max`

Đề nói "đảm bảo thắng **bất kể** đối thủ chọn số nào". Đối thủ có thể là kẻ xấu chọn số khiến ta tốn nhiều tiền nhất. Vì thế khi ta đoán `k` và sai, có **hai khả năng**: số thật nằm bên **trái** hoặc bên **phải**. Ta phải chuẩn bị tiền cho khả năng **tốn kém hơn**, nên dùng `max(chi_phi_trai, chi_phi_phai)`.

In [2]:
chi_phi_trai = 3
chi_phi_phai = 8
print("Phải chuẩn bị:", max(chi_phi_trai, chi_phi_phai))      # 8: lấy trường hợp xấu hơn

Phải chuẩn bị: 8


### 3.2. "Tối thiểu của tối đa" — `min` bao quanh `max` (minimax)

Ta được **chọn** số để đoán (`k`), còn đối thủ chọn nhánh. Với mỗi `k`, "chi phí xấu nhất" là `k + max(trái, phải)`. Ta chọn `k` làm chi phí xấu nhất **nhỏ nhất**. Đó là cấu trúc **minimax**: `min` (ta chọn) bọc ngoài `max` (đối thủ chọn).

`min` và `max` trong Python nhận cả **generator expression** (biểu thức sinh trong ngoặc, không tạo list trung gian).

In [3]:
phuong_an = {"A": (3, 8), "B": (5, 5), "C": (9, 1)}                # (chi_phi_trai, chi_phi_phai) cho 3 cách đoán
xau_nhat = {ten: max(t, p) for ten, (t, p) in phuong_an.items()}
print(xau_nhat)                                                     # {'A': 8, 'B': 5, 'C': 9}
print("Chọn:", min(xau_nhat, key=xau_nhat.get))                     # B: xấu nhất của B nhỏ nhất
print(min(max(t, p) for t, p in phuong_an.values()))                # 5 (dùng generator)

{'A': 8, 'B': 5, 'C': 9}
Chọn: B
5


### 3.3. Đoạn `[lo, hi]` và đoán trong đoạn

Sau mỗi lần đoán sai, tập số có thể là một **đoạn liên tiếp** nhỏ hơn: đoán `k` trong `[lo, hi]` thì còn `[lo, k-1]` (nếu số thật nhỏ hơn) hoặc `[k+1, hi]` (nếu lớn hơn). Đoạn **rỗng** (`lo > hi`) hoặc có **một phần tử** (`lo == hi`) thì **không tốn tiền**: không còn gì để sai (đoạn 1 phần tử: ta đoán trúng luôn).

In [4]:
lo, hi, k = 1, 10, 7
print("Đoán", k, "trong", (lo, hi))
print("  nếu số nhỏ hơn: còn", (lo, k - 1))
print("  nếu số lớn hơn: còn", (k + 1, hi))
print("Độ dài đoạn [8,10]:", 10 - 8 + 1)

Đoán 7 trong (1, 10)
  nếu số nhỏ hơn: còn (1, 6)
  nếu số lớn hơn: còn (8, 10)
Độ dài đoạn [8,10]: 3


### 3.4. Bảng 2 chiều (list lồng nhau) và cái bẫy `[[0]*m]*n`

Ta sẽ lưu `dp[i][j]` = đáp án cho đoạn `[i, j]` trong một list lồng nhau. **Cách tạo đúng:** `[[0] * m for _ in range(n)]`. **Cách sai:** `[[0] * m] * n` — nó lặp lại **cùng một list** `n` lần (các hàng là cùng một đối tượng), sửa một hàng sẽ sửa tất cả.

In [5]:
sai = [[0] * 3] * 2
sai[0][1] = 9
print("Sai :", sai)                         # [[0, 9, 0], [0, 9, 0]] - hai hàng cùng một list!

dung = [[0] * 3 for _ in range(2)]
dung[0][1] = 9
print("Đúng:", dung)                        # [[0, 9, 0], [0, 0, 0]]

Sai : [[0, 9, 0], [0, 9, 0]]
Đúng: [[0, 9, 0], [0, 0, 0]]


### 3.5. `functools.lru_cache` — nhớ kết quả đệ quy

`@functools.lru_cache(maxsize=None)` đặt trên một hàm: lần đầu gọi với một bộ tham số thì tính và **nhớ**; lần sau gọi lại **trả ngay** kết quả đã nhớ. Rất hợp cho đệ quy có các bài toán con **lặp lại**, ví dụ Fibonacci.

In [6]:
lan_goi = 0

@functools.lru_cache(maxsize=None)
def fib(n):
    global lan_goi
    lan_goi += 1
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(30), "- số lần thân hàm thực sự chạy:", lan_goi)   # 832040, chỉ 31 lần

832040 - số lần thân hàm thực sự chạy: 31


### 3.6. Vòng lặp theo độ dài đoạn

Với bảng `dp[i][j]` (đoạn `[i, j]`), muốn tính một đoạn ta cần các đoạn **ngắn hơn** nằm bên trong. Vậy ta duyệt **độ dài `length` từ nhỏ đến lớn**, với mỗi độ dài duyệt `i` rồi suy ra `j = i + length - 1`.

In [7]:
n = 4
for length in range(1, n + 1):
    cac_doan = [(i, i + length - 1) for i in range(1, n - length + 2)]
    print("độ dài", length, "->", cac_doan)

độ dài 1 -> [(1, 1), (2, 2), (3, 3), (4, 4)]
độ dài 2 -> [(1, 2), (2, 3), (3, 4)]
độ dài 3 -> [(1, 3), (2, 4)]
độ dài 4 -> [(1, 4)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Gọi `f(lo, hi)` là số tiền tối thiểu để chắc thắng khi số thật nằm trong `[lo, hi]`.

- **Đoạn 1 số** (`lo == hi`): đoán trúng ngay, `f = 0`. **Đoạn rỗng:** `f = 0`.
- **`n = 2`**, đoạn `[1,2]`: đoán `1` → sai thì mất `1` rồi biết là `2`: tổng `1`. Đoán `2` → sai thì mất `2`. Chọn `1`. `f(1,2) = 1`. Nhận xét: với 2 số, nên đoán số **nhỏ**.
- **`n = 3`**, `[1,3]`:
  - đoán `1`: `1 + max(f(rỗng), f(2,3))` = `1 + max(0, 2)` = `3` (vì `f(2,3)` = đoán `2` trong 2 số `2,3` mất `2`).
  - đoán `2`: `2 + max(f(1,1), f(3,3))` = `2 + 0` = `2`.
  - đoán `3`: `3 + max(f(1,2), 0)` = `3 + 1` = `4`.
  - Chọn `2` → `f(1,3) = 2`. Đoán số **ở giữa** để hai bên đều "chặn" ngay.
- **`n = 4`**, `[1,4]`: đoán `2`: `2 + max(f(1,1)=0, f(3,4)=3)` = `5`; đoán `3`: `3 + max(f(1,2)=1, f(4,4)=0)` = `4`; đoán `1`: `1 + f(2,4) = 1 + 3 = 4`; đoán `4`: `4 + f(1,3) = 6`. Tốt nhất `4`.

Nhận xét thú vị: số nên đoán **không phải đúng trung điểm** mà **lệch về phía lớn**, vì đoán số lớn thì phải trả nhiều tiền hơn khi sai; ta muốn tránh đoán sai ở các số lớn.

In [8]:
# Tính tay bằng code nhỏ để kiểm tra các con số ở trên (chưa cần thuật toán tổng quát)
f = {}                                           # f[(lo, hi)]
f[(1, 1)] = f[(2, 2)] = f[(3, 3)] = f[(4, 4)] = 0
f[(1, 2)] = min(1 + f[(2, 2)], 2 + f[(1, 1)])    # đoán 1 hoặc 2
f[(2, 3)] = min(2 + f[(3, 3)], 3 + f[(2, 2)])
f[(3, 4)] = min(3 + f[(4, 4)], 4 + f[(3, 3)])
print("f(1,2) =", f[(1, 2)], " f(2,3) =", f[(2, 3)], " f(3,4) =", f[(3, 4)])      # 1 2 3

f[(1, 3)] = min(1 + f[(2, 3)], 2 + max(f[(1, 1)], f[(3, 3)]), 3 + f[(1, 2)])
print("f(1,3) =", f[(1, 3)])                                                         # 2
f[(2, 4)] = min(2 + f[(3, 4)], 3 + max(f[(2, 2)], f[(4, 4)]), 4 + f[(2, 3)])
print("f(2,4) =", f[(2, 4)])                                                         # 3
f[(1, 4)] = min(1 + f[(2, 4)], 2 + max(f[(1, 1)], f[(3, 4)]), 3 + max(f[(1, 2)], f[(4, 4)]), 4 + f[(1, 3)])
print("f(1,4) =", f[(1, 4)])                                                         # 4

f(1,2) = 1  f(2,3) = 2  f(3,4) = 3
f(1,3) = 2
f(2,4) = 3
f(1,4) = 4


## 5. Cách 1 — Brute Force / baseline: đệ quy vét cạn

**Ý tưởng:** thử **mọi số `k`** trong `[lo, hi]` làm lần đoán đầu tiên. Với mỗi `k`, chi phí xấu nhất là `k + max(f(lo, k-1), f(k+1, hi))`. Lấy `min` qua mọi `k`.

```
f(lo, hi) = 0                                          nếu lo >= hi
f(lo, hi) = min( k + max(f(lo,k-1), f(k+1,hi)) )       với k = lo..hi
```

Đây là cách trực tiếp nhất: dịch đề thành đệ quy.

In [9]:
def brute(lo, hi):
    if lo >= hi:
        return 0
    best = float("inf")
    for k in range(lo, hi + 1):
        cost = k + max(brute(lo, k - 1), brute(k + 1, hi))
        best = min(best, cost)
    return best

for n, expected in examples:
    print(n, brute(1, n), expected)
    assert brute(1, n) == expected

10 16 16
1 0 0
2 1 1


**Độ phức tạp:** gọi `T(m)` là số lần gọi khi đoạn có `m` số. `T(m) = 1 + Σ_k ( T(k-1) + T(m-k) )`, tăng cỡ **hàm mũ** (khoảng `O(2^n)` đến tệ hơn). Không gian: độ sâu đệ quy `O(n)`.

**Nút thắt:** cùng một đoạn `[lo, hi]` bị tính đi tính lại rất nhiều lần (đoạn `[3,5]` xuất hiện trong cây đệ quy của `[1,5]`, `[1,6]`, `[2,5]`...). Hãy đo số lần gọi.

In [10]:
dem = {"goi": 0}

def brute_dem(lo, hi):
    dem["goi"] += 1
    if lo >= hi:
        return 0
    return min(k + max(brute_dem(lo, k - 1), brute_dem(k + 1, hi)) for k in range(lo, hi + 1))

for n in (4, 6, 8, 10, 12):
    dem["goi"] = 0
    brute_dem(1, n)
    print(f"n = {n:>2}: {dem['goi']:>9} lần gọi")

n =  4:        45 lần gọi
n =  6:       405 lần gọi
n =  8:      3645 lần gọi
n = 10:     32805 lần gọi
n = 12:    295245 lần gọi


Số lần gọi nhân lên rất nhanh (đúng 3 lần mỗi khi `n` tăng thêm 1 trong dãy trên: 45, 405, 3645, ...: nhân 9 sau mỗi 2 bước). Với `n = 200` là không thể. Nhưng **số đoạn khác nhau** chỉ khoảng `n²/2` — rất ít. Đó là gợi ý cho bước đột phá.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Bài toán con lặp lại:** đáp án của `[lo, hi]` chỉ phụ thuộc `lo` và `hi`, không phụ thuộc cách ta đã tới đây. Vậy chỉ có `O(n²)` trạng thái `(lo, hi)`.
2. **Mỗi trạng thái tốn `O(n)`** để thử mọi `k`. Tổng: `O(n²) × O(n) = O(n³)`; với `n = 200` là khoảng `200³/6 ≈ 1.3 triệu` phép tính: thoải mái.
3. Có hai cách tổ chức: **top-down** (đệ quy + nhớ) hoặc **bottom-up** (điền bảng theo độ dài đoạn tăng dần).
4. Có thể **cắt nhánh** vì chi phí `k + ...` luôn `>= k`: khi `k` đã lớn hơn đáp án tốt nhất hiện có thì thử tiếp vô ích.

## 7. Các cách tiếp theo

### Cách 2 — Đệ quy có nhớ (top-down memo)

**Giải thích từng đoạn:**

- `@functools.lru_cache(maxsize=None)` bọc hàm `solve(lo, hi)` để mỗi cặp `(lo, hi)` chỉ tính một lần.
- Nếu `lo >= hi`: trả `0`.
- Ngược lại thử mọi `k` trong `[lo, hi]`; chi phí `k + max(solve(lo, k-1), solve(k+1, hi))`; lấy `min`.

In [11]:
def memo_demo(n):
    @functools.lru_cache(maxsize=None)
    def solve(lo, hi):
        if lo >= hi:
            return 0
        return min(k + max(solve(lo, k - 1), solve(k + 1, hi)) for k in range(lo, hi + 1))
    return solve(1, n)

for n, expected in examples:
    print(n, memo_demo(n), expected)
    assert memo_demo(n) == expected
print("n = 200:", memo_demo(200))

10 16 16
1 0 0
2 1 1


n = 200: 952


**Code nộp LeetCode (Cách 2):** ta bọc `lru_cache` bên trong method. Lưu ý độ sâu đệ quy tối đa khoảng `n = 200` — nằm trong giới hạn mặc định `1000` của Python.

In [12]:
class SolutionMemo:
    def getMoneyAmount(self, n: int) -> int:
        @functools.lru_cache(maxsize=None)
        def solve(lo: int, hi: int) -> int:
            if lo >= hi:
                return 0
            return min(k + max(solve(lo, k - 1), solve(k + 1, hi)) for k in range(lo, hi + 1))

        return solve(1, n)

print(SolutionMemo().getMoneyAmount(10), SolutionMemo().getMoneyAmount(200))

16 952


**Độ phức tạp (Cách 2):** `O(n³)` thời gian (`O(n²)` trạng thái × `O(n)` mỗi trạng thái), `O(n²)` bộ nhớ cho cache (+ `O(n)` ngăn xếp đệ quy).

### Cách 3 — Quy hoạch động bottom-up trên đoạn (chuẩn nhất)

**Giải thích từng đoạn:**

- `dp = [[0] * (n + 2) for _ in range(n + 2)]` — thừa 1 hàng/cột ở hai đầu để `dp[k+1][j]` (khi `k = j`, tức `dp[j+1][j]`) và `dp[i][k-1]` (khi `k = i`, tức `dp[i][i-1]`) **không tràn chỉ số**. Giá trị mặc định `0` đúng cho đoạn rỗng/1 số.
- Duyệt `length` từ `2` đến `n`; `i` từ `1` đến `n - length + 1`; `j = i + length - 1`.
- `dp[i][j] = min(k + max(dp[i][k-1], dp[k+1][j]) for k in range(i, j + 1))`.
- Đáp án `dp[1][n]`.

In [13]:
def bottom_up_demo(n):
    dp = [[0] * (n + 2) for _ in range(n + 2)]
    for length in range(2, n + 1):
        for i in range(1, n - length + 2):
            j = i + length - 1
            dp[i][j] = min(k + max(dp[i][k - 1], dp[k + 1][j]) for k in range(i, j + 1))
    return dp[1][n]

for n, expected in examples:
    print(n, bottom_up_demo(n), expected)
    assert bottom_up_demo(n) == expected

10 16 16
1 0 0
2 1 1


In [14]:
class SolutionDP:
    def getMoneyAmount(self, n: int) -> int:
        dp = [[0] * (n + 2) for _ in range(n + 2)]
        for length in range(2, n + 1):
            for i in range(1, n - length + 2):
                j = i + length - 1
                dp[i][j] = min(k + max(dp[i][k - 1], dp[k + 1][j]) for k in range(i, j + 1))
        return dp[1][n]

print(SolutionDP().getMoneyAmount(10), SolutionDP().getMoneyAmount(200))

16 952


**Độ phức tạp (Cách 3):** `O(n³)` thời gian, `O(n²)` bộ nhớ. Không đệ quy nên không lo giới hạn ngăn xếp.

### Cách 4 — Đệ quy có nhớ + cắt nhánh (branch and bound)

**Ý tưởng khác:** chi phí của lần đoán `k` luôn `>= k` (vì ta trả `k` khi sai). Nếu ta duyệt `k` **tăng dần** và đã có chi phí tốt nhất `best`, thì khi gặp `k >= best` thì mọi `k` lớn hơn đều `>= best`: **dừng** vòng lặp, khỏi thử tiếp. Với đoạn ngắn hoặc số nhỏ cách này có hiệu quả, nhưng với đoạn gồm các số lớn (ví dụ `[150, 200]`) thì `best` luôn lớn hơn `k` nên ít cắt được. Nó vẫn **đúng**, chỉ là tối ưu nhỏ (và đo thực tế cho thấy gần như không giúp gì ở `n = 200`).

**Giải thích từng đoạn:** thử `k` từ `lo` đến `hi`; `if k >= best: break`; ngược lại tính chi phí và cập nhật `best`.

In [15]:
def pruned_demo(n):
    @functools.lru_cache(maxsize=None)
    def solve(lo, hi):
        if lo >= hi:
            return 0
        best = float("inf")
        for k in range(lo, hi + 1):
            if k >= best:                     # mọi k sau đều tốn >= k >= best: dừng
                break
            best = min(best, k + max(solve(lo, k - 1), solve(k + 1, hi)))
        return best
    return solve(1, n)

for n, expected in examples:
    assert pruned_demo(n) == expected
print("pruned n=10:", pruned_demo(10), " n=200:", pruned_demo(200))

pruned n=10: 16  n=200: 952


In [16]:
class SolutionPruned:
    def getMoneyAmount(self, n: int) -> int:
        @functools.lru_cache(maxsize=None)
        def solve(lo: int, hi: int) -> int:
            if lo >= hi:
                return 0
            best = float("inf")
            for k in range(lo, hi + 1):
                if k >= best:
                    break
                best = min(best, k + max(solve(lo, k - 1), solve(k + 1, hi)))
            return best

        return solve(1, n)

print(SolutionPruned().getMoneyAmount(10), SolutionPruned().getMoneyAmount(200))

16 952


**Độ phức tạp (Cách 4):** trường hợp xấu nhất vẫn `O(n³)` thời gian, `O(n²)` bộ nhớ; đo thực tế ở mục 16 cho thấy gần như không nhanh hơn Cách 2 (với `n = 200` hầu như không cắt được gì).

> Lưu ý: nhiều bạn nghĩ có thể giảm xuống `O(n²)` bằng "tối ưu Knuth" (vị trí đoán tốt nhất đơn điệu). Ở mục 15 ta sẽ **kiểm chứng bằng code** và thấy ý tưởng đó **không đúng** với bài này — đừng áp dụng bừa.

## 8. TRACE từng bước

Ta trace **Cách 3 (bottom-up)** với `n = 5`. Bảng `dp[i][j]` (hàng `i`, cột `j`, chỉ `i <= j`). Quy ước: `dp[i][i] = 0`.

| length | đoạn `[i,j]` | thử các `k` (chi phí `k+max(trái,phải)`) | `dp[i][j]` |
|---|---|---|---|
| 2 | [1,2] | k=1: 1+max(0,0)=1; k=2: 2+0=2 | 1 |
| 2 | [2,3] | k=2: 2; k=3: 3 | 2 |
| 2 | [3,4] | 3; 4 | 3 |
| 2 | [4,5] | 4; 5 | 4 |
| 3 | [1,3] | k=1: 1+2=3; k=2: 2+0=2; k=3: 3+1=4 | 2 |
| 3 | [2,4] | k=2: 2+3=5; k=3: 3+0=3; k=4: 4+2=6 | 3 |
| 3 | [3,5] | k=3: 3+4=7; k=4: 4+0=4; k=5: 5+3=8 | 4 |
| 4 | [1,4] | k=1: 1+3=4; k=2: 2+3=5; k=3: 3+1=4; k=4: 4+2=6 | 4 |
| 4 | [2,5] | k=2: 2+4=6; k=3: 3+4=7; k=4: 4+2=6; k=5: 5+3=8 | 6 |
| 5 | [1,5] | k=1: 1+6=7; k=2: 2+4=6; k=3: 3+max(1,4)=7; k=4: 4+max(2,0)=6; k=5: 5+4=9 | 6 |

Hàm `trace_dp` dưới đây in đúng những dòng này.

In [17]:
def trace_dp(n):
    dp = [[0] * (n + 2) for _ in range(n + 2)]
    for length in range(2, n + 1):
        print(f"--- length = {length} ---")
        for i in range(1, n - length + 2):
            j = i + length - 1
            options = []
            for k in range(i, j + 1):
                trai, phai = dp[i][k - 1], dp[k + 1][j]
                cost = k + max(trai, phai)
                options.append((k, trai, phai, cost))
            dp[i][j] = min(c for _, _, _, c in options)
            chi_tiet = "; ".join(f"k={k}: {k}+max({t},{p})={c}" for k, t, p, c in options)
            print(f"dp[{i}][{j}] = {dp[i][j]}   <-  {chi_tiet}")
    print("Đáp án =", dp[1][n])
    return dp[1][n]

assert trace_dp(5) == 6

--- length = 2 ---
dp[1][2] = 1   <-  k=1: 1+max(0,0)=1; k=2: 2+max(0,0)=2
dp[2][3] = 2   <-  k=2: 2+max(0,0)=2; k=3: 3+max(0,0)=3
dp[3][4] = 3   <-  k=3: 3+max(0,0)=3; k=4: 4+max(0,0)=4
dp[4][5] = 4   <-  k=4: 4+max(0,0)=4; k=5: 5+max(0,0)=5
--- length = 3 ---
dp[1][3] = 2   <-  k=1: 1+max(0,2)=3; k=2: 2+max(0,0)=2; k=3: 3+max(1,0)=4
dp[2][4] = 3   <-  k=2: 2+max(0,3)=5; k=3: 3+max(0,0)=3; k=4: 4+max(2,0)=6
dp[3][5] = 4   <-  k=3: 3+max(0,4)=7; k=4: 4+max(0,0)=4; k=5: 5+max(3,0)=8
--- length = 4 ---
dp[1][4] = 4   <-  k=1: 1+max(0,3)=4; k=2: 2+max(0,3)=5; k=3: 3+max(1,0)=4; k=4: 4+max(2,0)=6
dp[2][5] = 6   <-  k=2: 2+max(0,4)=6; k=3: 3+max(0,4)=7; k=4: 4+max(2,0)=6; k=5: 5+max(3,0)=8
--- length = 5 ---
dp[1][5] = 6   <-  k=1: 1+max(0,6)=7; k=2: 2+max(0,4)=6; k=3: 3+max(1,4)=7; k=4: 4+max(2,0)=6; k=5: 5+max(4,0)=9
Đáp án = 6


In [18]:
# In bảng dp dạng ma trận (hàng i, cột j) để nhìn "hình tam giác trên"
def bang_dp(n):
    dp = [[0] * (n + 2) for _ in range(n + 2)]
    for length in range(2, n + 1):
        for i in range(1, n - length + 2):
            j = i + length - 1
            dp[i][j] = min(k + max(dp[i][k - 1], dp[k + 1][j]) for k in range(i, j + 1))
    return dp

n = 6
dp = bang_dp(n)
print("      " + " ".join(f"j={j:<2}" for j in range(1, n + 1)))
for i in range(1, n + 1):
    print(f"i={i}   " + " ".join(f"{dp[i][j] if j >= i else '.':<4}" for j in range(1, n + 1)))

      j=1  j=2  j=3  j=4  j=5  j=6 
i=1   0    1    2    4    6    8   
i=2   .    0    2    3    6    8   
i=3   .    .    0    3    4    8   
i=4   .    .    .    0    4    5   
i=5   .    .    .    .    0    5   
i=6   .    .    .    .    .    0   


Đọc bảng: góc trên phải `dp[1][n]` là đáp án. Đường chéo chính toàn `0`. Giá trị tăng khi đi sang phải và giảm khi đi xuống (đoạn càng dài càng đắt; đoạn cùng độ dài nhưng gồm các số **lớn hơn** thì đắt hơn).

## 9. Code nộp LeetCode

Ta nộp Cách 3 (bottom-up), không đệ quy, ổn định.

In [19]:
class Solution:
    def getMoneyAmount(self, n: int) -> int:
        # dp[i][j]: số tiền tối thiểu để chắc thắng khi số thật nằm trong [i, j]
        dp = [[0] * (n + 2) for _ in range(n + 2)]
        for length in range(2, n + 1):
            for i in range(1, n - length + 2):
                j = i + length - 1
                dp[i][j] = min(k + max(dp[i][k - 1], dp[k + 1][j]) for k in range(i, j + 1))
        return dp[1][n]

sol = Solution()
for n, expected in examples:
    got = sol.getMoneyAmount(n)
    print(n, got, expected, "OK" if got == expected else "SAI")
    assert got == expected

10 16 16 OK
1 0 0 OK
2 1 1 OK


## 10. Vì sao thuật toán đúng

1. **Tính đúng của công thức.** Khi số thật nằm trong `[lo, hi]` và ta đoán `k`: nếu đúng thì chi phí `0`; nếu sai ta trả `k` rồi quay về bài toán tương tự trên `[lo, k-1]` hoặc `[k+1, hi]`. Đối thủ chọn nhánh xấu hơn nên chi phí xấu nhất là `k + max(f(trái), f(phải))`. Ta chọn `k` tối ưu nên lấy `min`. Đây là **cấu trúc con tối ưu**.
2. **Vì sao chỉ cần đoạn liên tiếp.** Mọi thông tin ta biết sau các lần đoán chỉ là "số thật nằm trong khoảng nào đó"; không có thông tin nào khác, nên trạng thái đủ là một đoạn `[lo, hi]`.
3. **Vì sao có thể điền theo độ dài.** `f(lo, hi)` chỉ cần các đoạn con **thật sự ngắn hơn** (`[lo, k-1]` và `[k+1, hi]` đều ngắn hơn `hi-lo+1`), nên khi duyệt độ dài tăng dần thì chúng đã có sẵn.
4. **Cơ sở.** Đoạn rỗng và đoạn 1 số không tốn tiền, trùng với giá trị mặc định `0`.

Ta cũng kiểm chứng bằng một oracle **hoàn toàn khác** ở mục 12 (liệt kê mọi cây tìm kiếm nhị phân = mọi chiến lược).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Đệ quy vét cạn | cỡ hàm mũ | `O(n)` ngăn xếp |
| 2. Memo top-down | `O(n³)` | `O(n²)` |
| 3. Bottom-up DP | `O(n³)` | `O(n²)` |
| 4. Memo + cắt nhánh | `O(n³)` (thực tế gần bằng Cách 2) | `O(n²)` |

Với `n = 200`: số cặp `(i, j)` ≈ `20.000`, mỗi cặp thử tối đa `200` giá trị `k`; trung bình chỉ khoảng `n/3` nên tổng cỡ `1.3 triệu` phép tính — chạy trong chưa đầy 1 giây.

In [20]:
t0 = time.perf_counter()
print(Solution().getMoneyAmount(200))
print("Thời gian n = 200: %.3f giây" % (time.perf_counter() - t0))

952
Thời gian n = 200: 0.117 giây


## 12. Bộ test

Gồm: ví dụ đề, biên, phản ví dụ, đối chiếu các cách với nhau, và đối chiếu với **oracle độc lập**: mỗi chiến lược đoán chính là một **cây tìm kiếm nhị phân** (gốc = lần đoán đầu, cây con trái/phải = các lần đoán sau), chi phí xấu nhất của chiến lược = **max, trên mọi nút, của tổng các số tổ tiên** (các số đã đoán sai trước khi đoán trúng nút đó). Ta liệt kê **mọi** BST trên `1..n` (số Catalan) rồi lấy min.

In [21]:
def all_bsts(lo, hi):
    """Sinh mọi cây tìm kiếm nhị phân trên [lo, hi]: mỗi cây là (gốc, trái, phải) hoặc None."""
    if lo > hi:
        yield None
        return
    for root in range(lo, hi + 1):
        for left in all_bsts(lo, root - 1):
            for right in all_bsts(root + 1, hi):
                yield (root, left, right)

def worst_cost(tree, tong_to_tien=0):
    """Chi phí xấu nhất = max qua các nút của (tổng các nút tổ tiên đã đoán sai)."""
    if tree is None:
        return 0
    root, left, right = tree
    return max(tong_to_tien, worst_cost(left, tong_to_tien + root), worst_cost(right, tong_to_tien + root))

def oracle_bst(n):
    return min(worst_cost(t) for t in all_bsts(1, n))

print([oracle_bst(n) for n in range(1, 10)])
print([Solution().getMoneyAmount(n) for n in range(1, 10)])
for n in range(1, 10):
    assert oracle_bst(n) == Solution().getMoneyAmount(n)
print("Oracle BST khớp lời giải với n = 1..9")

[0, 1, 2, 4, 6, 8, 10, 12, 14]
[0, 1, 2, 4, 6, 8, 10, 12, 14]
Oracle BST khớp lời giải với n = 1..9


### 12.1. Các trường hợp biên và giá trị đã biết

In [22]:
known = {1: 0, 2: 1, 3: 2, 4: 4, 5: 6, 6: 8, 7: 10, 8: 12, 9: 14, 10: 16, 20: 49, 50: 172, 100: 400, 200: 952}
for n, expected in known.items():
    got = Solution().getMoneyAmount(n)
    assert got == expected, (n, got, expected)
print("Các giá trị đã biết đều khớp:", known)

Các giá trị đã biết đều khớp: {1: 0, 2: 1, 3: 2, 4: 4, 5: 6, 6: 8, 7: 10, 8: 12, 9: 14, 10: 16, 20: 49, 50: 172, 100: 400, 200: 952}


### 12.2. Phản ví dụ cho các cách làm sai

Hai suy nghĩ rất hấp dẫn nhưng **sai**:

1. **Chia đôi mãi** (đoán trung điểm): nghe hợp lý vì là binary search.
2. **Luôn đoán số nhỏ nhất** (quét tuần tự từ 1): nghe an toàn vì số nhỏ rẻ.

In [23]:
def chia_doi_sai(n):
    @functools.lru_cache(maxsize=None)
    def go(lo, hi):
        if lo >= hi:
            return 0
        k = (lo + hi) // 2
        return k + max(go(lo, k - 1), go(k + 1, hi))
    return go(1, n)

def quet_tuan_tu_sai(n):
    # đoán 1, 2, 3, ..., n-1: trường hợp xấu nhất là số thật n: tốn 1 + 2 + ... + (n-1)
    return n * (n - 1) // 2

for n in (4, 10, 20):
    print(f"n={n:>2}: đúng={Solution().getMoneyAmount(n):>3}  chia đôi={chia_doi_sai(n):>3}  quét tuần tự={quet_tuan_tu_sai(n):>3}")
assert chia_doi_sai(10) > 16 and quet_tuan_tu_sai(10) > 16

n= 4: đúng=  4  chia đôi=  5  quét tuần tự=  6
n=10: đúng= 16  chia đôi= 22  quét tuần tự= 45
n=20: đúng= 49  chia đôi= 62  quét tuần tự=190


Với `n = 10`: đáp án đúng `16`, chia đôi cho `22`, quét tuần tự cho `45`. Chiến lược tối ưu đoán lệch về phía **lớn** (ví dụ đoán `7` chứ không phải `5`).

### 12.3. Đối chiếu các cách với nhau (nhiều input)

In [24]:
def brute_check(n):
    return brute(1, n)

for n in range(1, 13):
    a = brute_check(n)
    b = SolutionMemo().getMoneyAmount(n)
    c = SolutionDP().getMoneyAmount(n)
    d = SolutionPruned().getMoneyAmount(n)
    assert a == b == c == d, (n, a, b, c, d)

for n in range(1, 201):
    b = SolutionMemo().getMoneyAmount(n)
    c = SolutionDP().getMoneyAmount(n)
    d = SolutionPruned().getMoneyAmount(n)
    assert b == c == d, (n, b, c, d)
print("4 cách khớp nhau với n = 1..12 (có vét cạn) và n = 1..200 (3 cách DP)")

4 cách khớp nhau với n = 1..12 (có vét cạn) và n = 1..200 (3 cách DP)


In [25]:
# Ngẫu nhiên có seed: n ngẫu nhiên trong ràng buộc, kiểm tra tính chất
random.seed(375)
sol = Solution()
for _ in range(30):
    n = random.randint(1, 200)
    v = sol.getMoneyAmount(n)
    assert v == SolutionMemo().getMoneyAmount(n)
    assert v >= 0
    assert sol.getMoneyAmount(n + 1) >= v if n < 200 else True        # tính đơn điệu không giảm theo n
    assert v <= n * (n - 1) // 2                                       # không tệ hơn quét tuần tự
print("Ngẫu nhiên: đúng, đơn điệu, không vượt n(n-1)/2")

Ngẫu nhiên: đúng, đơn điệu, không vượt n(n-1)/2


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Thay `max` bằng `min` (hoặc cộng hai nhánh)

Đề yêu cầu **đảm bảo** thắng nên phải lo cho nhánh xấu hơn; dùng `min` là tính cho trường hợp "may mắn".

In [26]:
def sai_min(n):
    dp = [[0] * (n + 2) for _ in range(n + 2)]
    for length in range(2, n + 1):
        for i in range(1, n - length + 2):
            j = i + length - 1
            dp[i][j] = min(k + min(dp[i][k - 1], dp[k + 1][j]) for k in range(i, j + 1))   # SAI: min thay vì max
    return dp[1][n]

print("n=10: sai =", sai_min(10), "| đúng =", Solution().getMoneyAmount(10))

n=10: sai = 1 | đúng = 16


Vì sao sai: `min(trái, phải)` giả định đối thủ "tốt bụng" chọn nhánh rẻ hơn. Đối thủ trong đề là xấu nhất có thể → phải `max`. **Đúng:** `k + max(...)`.

### Lỗi 2: Chỉ số `dp[k+1][j]` bị tràn khi bảng quá nhỏ

Khi `k = j`, ta cần `dp[j+1][j]`; nếu `dp` chỉ có `n + 1` hàng thì `j + 1 = n + 1` còn được, nhưng nếu dùng `n` hàng thì `IndexError`.

In [27]:
def sai_chi_so(n):
    dp = [[0] * (n + 1) for _ in range(n + 1)]       # SAI: thiếu cột/hàng dự phòng
    for length in range(2, n + 1):
        for i in range(1, n - length + 2):
            j = i + length - 1
            dp[i][j] = min(k + max(dp[i][k - 1], dp[k + 1][j]) for k in range(i, j + 1))
    return dp[1][n]

try:
    sai_chi_so(5)
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


**Đúng:** cấp `n + 2` hàng và cột (chỉ số hợp lệ `0..n+1`), như ở lời giải chuẩn.

### Lỗi 3: Tạo bảng bằng `[[0] * m] * n`

Các hàng cùng một list nên ghi `dp[i][j]` làm đổi cả các hàng khác. Kết quả sai âm thầm, không báo lỗi.

In [28]:
def sai_alias(n):
    dp = [[0] * (n + 2)] * (n + 2)                   # SAI: các hàng là cùng một list
    for length in range(2, n + 1):
        for i in range(1, n - length + 2):
            j = i + length - 1
            dp[i][j] = min(k + max(dp[i][k - 1], dp[k + 1][j]) for k in range(i, j + 1))
    return dp[1][n]

print("n=10: sai =", sai_alias(10), "| đúng =", Solution().getMoneyAmount(10))

n=10: sai = 61 | đúng = 16


**Đúng:** `[[0] * (n + 2) for _ in range(n + 2)]`.

### Lỗi 4: Đặt sai cơ sở: `dp[i][i] = i`

Nhiều bạn nghĩ "đoạn một số `i` thì tốn `i`". Nhưng nếu chỉ còn **một** khả năng, ta đoán trúng ngay và **không mất tiền**: `dp[i][i] = 0`.

In [29]:
def sai_co_so(n):
    dp = [[0] * (n + 2) for _ in range(n + 2)]
    for i in range(1, n + 1):
        dp[i][i] = i                                  # SAI: đoạn 1 số phải bằng 0
    for length in range(2, n + 1):
        for i in range(1, n - length + 2):
            j = i + length - 1
            dp[i][j] = min(k + max(dp[i][k - 1], dp[k + 1][j]) for k in range(i, j + 1))
    return dp[1][n]

print("n=10: sai =", sai_co_so(10), "| đúng =", Solution().getMoneyAmount(10))

n=10: sai = 26 | đúng = 16


### Lỗi 5: Duyệt sai thứ tự (không theo độ dài tăng dần)

Nếu duyệt `i` từ `1` lên `n` rồi `j` từ `i` đến `n`, thì khi tính `dp[i][j]` cần `dp[k+1][j]` (hàng sau) chưa được tính. Hãy theo độ dài, hoặc `i` giảm dần.

In [30]:
def sai_thu_tu(n):
    dp = [[0] * (n + 2) for _ in range(n + 2)]
    for i in range(1, n + 1):                         # SAI: i tăng dần -> dp[k+1][j] chưa sẵn sàng
        for j in range(i + 1, n + 1):
            dp[i][j] = min(k + max(dp[i][k - 1], dp[k + 1][j]) for k in range(i, j + 1))
    return dp[1][n]

print("n=10: sai =", sai_thu_tu(10), "| đúng =", Solution().getMoneyAmount(10))

n=10: sai = 1 | đúng = 16


**Đúng:** theo `length` tăng dần; hoặc `for i in range(n, 0, -1): for j in range(i + 1, n + 1)`.

## 14. Mẹo & nhận dạng pattern

- **Từ khóa nhận dạng:** "đảm bảo/bất kể", "tối thiểu cái xấu nhất", "min of max" → **minimax**. Hai bên đối kháng.
- **Có nhánh trái/phải sau mỗi bước chọn trên một dãy liên tiếp** → **DP trên đoạn** (interval DP), cùng họ với *Burst Balloons* (312), *Minimum Cost to Cut a Stick* (1547), *Matrix Chain Multiplication*, *Strange Printer*.
- **Khung chung của interval DP:** `dp[i][j] = tối ưu qua k trong [i, j] của ( chi phí k + kết hợp(dp[i][k-1], dp[k+1][j]) )`; duyệt theo độ dài.
- **Mẹo nhớ:** "chọn k làm lần đoán đầu → chia đôi đoạn". Hãy viết `dp[1][1] = 0`, `dp[1][2] = 1` ra giấy rồi mới code.
- **Kiểm tra nhanh bằng tay:** `n = 3` phải ra `2` (đoán 2). `n = 4` ra `4`.
- **Giới hạn `n ≤ 200`** gợi ý `O(n³)` là đủ.

## 15. Biến thể

### 15.1. Dựng lại chiến lược (nên đoán số nào đầu tiên)

Lưu `choice[i][j]` = `k` tốt nhất, rồi đọc lại.

In [31]:
def chien_luoc(n):
    dp = [[0] * (n + 2) for _ in range(n + 2)]
    choice = [[0] * (n + 2) for _ in range(n + 2)]
    for length in range(2, n + 1):
        for i in range(1, n - length + 2):
            j = i + length - 1
            best, bk = None, None
            for k in range(i, j + 1):
                c = k + max(dp[i][k - 1], dp[k + 1][j])
                if best is None or c < best:
                    best, bk = c, k
            dp[i][j], choice[i][j] = best, bk
    return dp, choice

dp, choice = chien_luoc(10)
print("Chi phí:", dp[1][10], "| đoán đầu tiên:", choice[1][10])
print("Nếu số > 7 → đoạn [8,10]: đoán", choice[8][10])
print("Nếu số < 7 → đoạn [1,6]: đoán", choice[1][6])
assert dp[1][10] == 16

Chi phí: 16 | đoán đầu tiên: 7
Nếu số > 7 → đoạn [8,10]: đoán 9
Nếu số < 7 → đoạn [1,6]: đoán 3


### 15.2. Tối ưu Knuth không áp dụng được (kiểm chứng)

Giả thuyết: vị trí tối ưu `opt[i][j]` thỏa `opt[i][j-1] <= opt[i][j] <= opt[i+1][j]` (như ở một số bài DP khác) để duyệt `k` hẹp lại, giảm còn `O(n²)`. Kiểm tra bằng code: nếu giả thuyết đúng thì kết quả phải bằng DP đầy đủ.

In [32]:
def knuth_thu(n):
    dp = [[0] * (n + 2) for _ in range(n + 2)]
    opt = [[0] * (n + 2) for _ in range(n + 2)]
    for i in range(1, n + 1):
        opt[i][i] = i
    for length in range(2, n + 1):
        for i in range(1, n - length + 2):
            j = i + length - 1
            best, bk = None, None
            for k in range(opt[i][j - 1], min(opt[i + 1][j], j) + 1):
                c = k + max(dp[i][k - 1], dp[k + 1][j])
                if best is None or c < best:
                    best, bk = c, k
            dp[i][j], opt[i][j] = best, bk
    return dp[1][n]

for n in (3, 5, 6, 10):
    print(n, "Knuth-thử:", knuth_thu(n), "| đúng:", Solution().getMoneyAmount(n))
assert knuth_thu(6) != Solution().getMoneyAmount(6)

3 Knuth-thử: 2 | đúng: 2
5 Knuth-thử: 6 | đúng: 6
6 Knuth-thử: 9 | đúng: 8
10 Knuth-thử: 16 | đúng: 16


Với `n = 6`, "tối ưu Knuth" cho `9` thay vì `8` — **sai**. Bài học: không phải interval DP nào cũng có tính chất đơn điệu của `opt`; luôn kiểm tra bằng oracle trước khi tin một tối ưu hóa.

### 15.3. Các biến thể khác

- **Chi phí `k²` thay vì `k`, hoặc chi phí cố định mỗi lần đoán:** chỉ đổi `k` trong công thức thành `cost(k)`.
- **Đoán số bị hạn chế** (chỉ được đoán số lẻ...): thu hẹp tập `k`.
- **Guess Number Higher or Lower** (LeetCode 374): mỗi lần đoán tốn 1 lượt, dùng chia đôi `O(log n)`; bài này là bản "có giá".
- **Chi phí kỳ vọng** (đối thủ chọn đều ngẫu nhiên) thay vì xấu nhất: `f = k-cost + trung bình có trọng số`, là bài toán khác hẳn (không còn `max`).
- **Cắt que (1547), Burst Balloons (312):** cùng khung interval DP.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đệ quy vét cạn | Thử mọi `k`, không nhớ | cỡ hàm mũ | `O(n)` | Chỉ để hiểu đề, kiểm chứng `n` rất nhỏ |
| 2. Memo top-down | Như 1 + `lru_cache` | `O(n³)` | `O(n²)` | Viết nhanh, ít lỗi chỉ số |
| 3. Bottom-up | Điền bảng theo độ dài | `O(n³)` | `O(n²)` | Chuẩn nhất, không đệ quy |
| 4. Memo + cắt nhánh | Dừng khi `k >= best` | `O(n³)` (gần bằng Cách 2) | `O(n²)` | Để học kỹ thuật cắt nhánh; lợi ích nhỏ ở bài này |

Cả ba cách DP đều qua được `n = 200` rất nhanh; chọn cách 3 để nộp.

In [33]:
def do_thoi_gian(fn, n=200, lan=3):
    t = time.perf_counter()
    for _ in range(lan):
        fn(n)
    return (time.perf_counter() - t) / lan

for ten, cls in [("memo", SolutionMemo), ("bottom-up", SolutionDP), ("pruned", SolutionPruned)]:
    print(f"{ten:<10}: {do_thoi_gian(cls().getMoneyAmount):.4f} giây / lần (n=200)")

memo      : 0.2636 giây / lần (n=200)


bottom-up : 0.1081 giây / lần (n=200)


pruned    : 0.2889 giây / lần (n=200)


## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `getMoneyAmount(3)`; (b) `getMoneyAmount(4)`; (c) `max(3, 8) + 2`; (d) giá trị `dp[2][3]` khi `n = 5`; (e) vì sao `dp[i][i] = 0`.

**Bài 2 (tính tay).** Tính `f(1,5)` bằng tay theo công thức (đã có bảng ở mục 8 — hãy tự tính lại **không nhìn**) và chỉ ra **số đầu tiên nên đoán**.

**Bài 3 (điền TODO).** Hoàn thành `money_todo(n)` theo Cách 3.

In [34]:
def money_todo(n):
    # TODO 1: tạo dp là bảng (n + 2) x (n + 2) toàn 0 (cẩn thận đừng dùng [[0]*m]*n)
    # TODO 2: for length in range(2, n + 1): for i ...: j = i + length - 1
    # TODO 3: dp[i][j] = min( k + max(dp[i][k-1], dp[k+1][j]) for k in range(i, j+1) )
    # TODO 4: return dp[1][n]
    return None

# KHÔNG gọi money_todo ở đây để "Run All" không lỗi (chưa điền xong).

**Bài 4 (tự viết lại).** Viết `money_memo(n)` bằng `lru_cache` **mà không nhìn lại** mục 7. Kiểm tra `n = 10 → 16`, `n = 100 → 400`.

**Bài 5 (biến thể / mini challenge).** Mỗi lần đoán sai số `x` ta mất `x * x` đô (bình phương). Viết `money_square(n)` và tính `money_square(10)`. Gợi ý: chỉ cần đổi `k` thành `k * k`. Thử so với phiên bản chi phí `k`: chiến lược đoán đầu tiên có thay đổi không?

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: (a) `2`, (b) `4`, (c) `10`, (d) `2`; (e) chỉ còn một khả năng nên đoán là trúng, không tốn gì. Bài 3: bắt đầu từ `dp = [[0] * (n + 2) for _ in range(n + 2)]`.
- **Mức 2 (vừa):** Bài 2: các `k` cho `[1,5]` lần lượt `7, 6, 7, 6, 9`; hai số đầu tiên tốt nhất là `2` hoặc `4`. Bài 4: `lru_cache` bọc hàm `solve(lo, hi)` với cơ sở `lo >= hi → 0`.
- **Mức 3 (sâu):** Bài 5: thay `k + max(...)` bằng `k * k + max(...)`. Vì chi phí tăng nhanh theo số lớn nên ta còn thích đoán số **nhỏ hơn** so với trước; hãy in `choice[1][n]` để so sánh.

## 19. Đáp án tham khảo cho bài tập

In [35]:
# Bài 1
assert Solution().getMoneyAmount(3) == 2
assert Solution().getMoneyAmount(4) == 4
assert max(3, 8) + 2 == 10
assert bang_dp(5)[2][3] == 2
print("Bài 1 OK")

# Bài 2: f(1,5) = 6, số đầu tiên nên đoán: 2 hoặc 4
cac_k = {k: k + max(bang_dp(5)[1][k - 1], bang_dp(5)[k + 1][5]) for k in range(1, 6)}
print("Chi phí theo k:", cac_k)
assert cac_k == {1: 7, 2: 6, 3: 7, 4: 6, 5: 9}
assert min(cac_k.values()) == 6

# Bài 3
def money_todo_ans(n):
    dp = [[0] * (n + 2) for _ in range(n + 2)]
    for length in range(2, n + 1):
        for i in range(1, n - length + 2):
            j = i + length - 1
            dp[i][j] = min(k + max(dp[i][k - 1], dp[k + 1][j]) for k in range(i, j + 1))
    return dp[1][n]

for n in range(1, 40):
    assert money_todo_ans(n) == Solution().getMoneyAmount(n)
print("Bài 3 OK")

# Bài 4
def money_memo(n):
    @functools.lru_cache(maxsize=None)
    def solve(lo, hi):
        if lo >= hi:
            return 0
        return min(k + max(solve(lo, k - 1), solve(k + 1, hi)) for k in range(lo, hi + 1))
    return solve(1, n)

assert money_memo(10) == 16 and money_memo(100) == 400
print("Bài 4 OK")

# Bài 5
def money_square(n):
    dp = [[0] * (n + 2) for _ in range(n + 2)]
    for length in range(2, n + 1):
        for i in range(1, n - length + 2):
            j = i + length - 1
            dp[i][j] = min(k * k + max(dp[i][k - 1], dp[k + 1][j]) for k in range(i, j + 1))
    return dp[1][n]

print("money_square(10) =", money_square(10))
assert money_square(1) == 0 and money_square(2) == 1 and money_square(3) == 4
print("Bài 5 OK")

Bài 1 OK
Chi phí theo k: {1: 7, 2: 6, 3: 7, 4: 6, 5: 9}
Bài 3 OK
Bài 4 OK
money_square(10) = 130
Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (interval DP minimax):**

```python
dp = [[0] * (n + 2) for _ in range(n + 2)]
for length in range(2, n + 1):
    for i in range(1, n - length + 2):
        j = i + length - 1
        dp[i][j] = min(k + max(dp[i][k - 1], dp[k + 1][j]) for k in range(i, j + 1))
return dp[1][n]
```

**Checklist:**

- [ ] Công thức dùng `min` ngoài, `max` trong (minimax), **không** phải `min`/`min` hay `+`.
- [ ] `dp[i][i] = 0` và đoạn rỗng `= 0` (mặc định `0`).
- [ ] Bảng kích thước `(n + 2) x (n + 2)`, tạo bằng list comprehension (không alias).
- [ ] Duyệt theo độ dài tăng dần (`length` từ `2`), `j = i + length - 1`.
- [ ] `k` chạy từ `i` đến `j` **bao gồm cả hai đầu**.
- [ ] Kiểm tra bằng tay `n = 1, 2, 3, 4` → `0, 1, 2, 4`; `n = 10` → `16`.
- [ ] Đúng signature `class Solution` / `getMoneyAmount(self, n: int) -> int`.